# Module 09 — Model Routing

> **SDKs:** `pydantic`, `dataclasses`, `typing`

| Part | Topic |
|------|-------|
| **1** | Capability Filtering — Model Registry and modality matching |
| **2** | Model Cascades — cost optimization with programmatic assertions |
| **3** | Fallbacks & Reliability — provider failover on 429/503 |


---
## Part 1 — Capability Filtering: The Model Registry

A `gpt-4o-mini` cannot process screenshots. A model registry allows the router to dynamically select a model that supports the required modalities.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, Literal
from pydantic import BaseModel

@dataclass
class ModelSpec:
    model_id: str
    provider: str
    modalities: list[str]      # ["text", "vision", "audio", "function_calling"]
    context_window: int
    cost_per_1k_input: float   # USD
    cost_per_1k_output: float
    max_throughput_rpm: int
    available: bool = True

MODEL_REGISTRY: list[ModelSpec] = [
    ModelSpec("gpt-4o-mini",      "openai",    ["text","function_calling"], 128000, 0.00015, 0.00060, 500),
    ModelSpec("gpt-4o",           "openai",    ["text","vision","function_calling","audio"], 128000, 0.002500, 0.010000, 200),
    ModelSpec("claude-3-haiku",   "anthropic", ["text","function_calling"], 200000, 0.000250, 0.001250, 400),
    ModelSpec("claude-3-5-sonnet","anthropic", ["text","vision","function_calling"], 200000, 0.003000, 0.015000, 100),
    ModelSpec("gemini-1.5-flash", "google",    ["text","vision","audio","function_calling"], 1000000, 0.000075, 0.000300, 1000),
    ModelSpec("gemini-1.5-pro",   "google",    ["text","vision","audio","function_calling"], 2000000, 0.001250, 0.005000, 200),
]

class ModelRouter:
    def __init__(self, registry: list[ModelSpec]):
        self.registry = registry

    def select(
        self,
        required_modalities: list[str],
        context_tokens: int,
        prefer_cheapest: bool = True,
    ) -> Optional[ModelSpec]:
        """
        1. Filter: only models that support ALL required modalities
        2. Filter: context window large enough
        3. Filter: available (not rate-limited or degraded)
        4. Sort by cost (cheapest first) or quality
        """
        eligible = [
            m for m in self.registry
            if all(mod in m.modalities for mod in required_modalities)
            and m.context_window >= context_tokens
            and m.available
        ]
        if not eligible:
            return None
        return min(eligible, key=lambda m: m.cost_per_1k_input) if prefer_cheapest else eligible[0]

router = ModelRouter(MODEL_REGISTRY)

requests = [
    (["text"],                    1000, "Simple text generation"),
    (["text","function_calling"], 5000, "Tool-calling pipeline"),
    (["text","vision"],           8000, "Screenshot analysis"),
    (["text","audio"],            2000, "Voice transcription"),
    (["text","vision","audio"],   3000, "Multimodal analysis"),
]

print("🎛️  Model Router: Capability Filtering Demo")
print("=" * 75)
print(f"  {'Request':<35} {'Selected Model':<25} {'Cost/1k in'}")
print(f"  {'─'*35} {'─'*25} {'─'*12}")

for modalities, ctx, label in requests:
    selected = router.select(modalities, ctx)
    if selected:
        print(f"  {label:<35} {selected.model_id:<25} ${selected.cost_per_1k_input:.5f}")
    else:
        print(f"  {label:<35} ❌ No eligible model in registry")


🎛️  Model Router: Capability Filtering Demo
  Request                             Selected Model            Cost/1k in
  ─────────────────────────────────── ───────────────────────── ────────────
  Simple text generation              gemini-1.5-flash          $0.00008
  Tool-calling pipeline               gemini-1.5-flash          $0.00008
  Screenshot analysis                 gemini-1.5-flash          $0.00008
  Voice transcription                 gemini-1.5-flash          $0.00008
  Multimodal analysis                 gemini-1.5-flash          $0.00008


---
## Part 2 — Model Cascades: Cheap → Expensive on Failure

Try a cheap model first. Run a programmatic assertion. Only if it fails, escalate to the expensive model. This can cut costs by 60-80% on simple tasks.

In [ ]:
import random, time
from dataclasses import dataclass

@dataclass
class CascadeResult:
    final_model: str
    attempts: int
    total_cost_usd: float
    output: str

def run_cascade(task: str, difficulty: str = "easy") -> CascadeResult:
    """
    Cascade: gpt-4o-mini → gpt-4o → claude-3-5-sonnet
    Assertion: output is valid JSON with required keys.
    """
    import json
    
    cascade = [
        ("gpt-4o-mini",       0.000150, 0.000600),
        ("gpt-4o",            0.002500, 0.010000),
        ("claude-3-5-sonnet", 0.003000, 0.015000),
    ]
    
    total_cost = 0.0
    attempts = 0
    tokens = len(task.split()) * 4   # rough estimate
    
    for model_id, cost_in, cost_out in cascade:
        attempts += 1
        total_cost += (tokens / 1000) * cost_in + (100 / 1000) * cost_out
        
        # Simulate: cheap models succeed on easy tasks, need escalation for hard
        success_rate = {"easy": 0.95, "medium": 0.65, "hard": 0.30}.get(difficulty, 0.5)
        if model_id != "gpt-4o-mini": success_rate = min(success_rate + 0.25, 0.97)
        
        print(f"  [{model_id}] Attempting task (attempt {attempts})...", end=" ")
        time.sleep(0.02)
        
        if random.random() < success_rate:
            output = json.dumps({"status": "success", "model": model_id, "task": task[:30]})
            print(f"✅  (cost so far: ${total_cost:.5f})")
            return CascadeResult(model_id, attempts, total_cost, output)
        else:
            print(f"❌  Assertion failed — escalating to next tier")

    return CascadeResult("none", attempts, total_cost, "FAILED")

random.seed(99)
print("💧  Model Cascade Demo")
print("=" * 60)

for difficulty in ["easy", "hard"]:
    print(f"\n  Task difficulty: {difficulty.upper()}")
    result = run_cascade(f"Extract JSON from this {difficulty} support ticket", difficulty)
    print(f"  → Final model: {result.final_model}  attempts: {result.attempts}  total_cost: ${result.total_cost:.5f}")


💧  Model Cascade Demo

  Task difficulty: EASY
  [gpt-4o-mini] Attempting task (attempt 1)... ✅  (cost so far: $0.00022)
  → Final model: gpt-4o-mini  attempts: 1  total_cost: $0.00022

  Task difficulty: HARD
  [gpt-4o-mini] Attempting task (attempt 1)... ❌  Assertion failed — escalating to next tier
  [gpt-4o] Attempting task (attempt 2)... ❌  Assertion failed — escalating to next tier
  [claude-3-5-sonnet] Attempting task (attempt 3)... ✅  (cost so far: $0.01822)
  → Final model: claude-3-5-sonnet  attempts: 3  total_cost: $0.01822


---
## Part 3 — Fallbacks & Reliability: Provider Failover

A single LLM provider will eventually return 429 (rate limit) or 503 (outage). The router must fall over to a secondary provider without user impact.

In [ ]:
import random, time
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class ProviderHealth:
    provider: str
    status: Literal["healthy", "degraded", "down"] = "healthy"
    error_rate: float = 0.0
    p99_latency_ms: float = 200.0

class ReliableRouter:
    """
    Maintains a priority-ordered list of providers.
    Skips providers that are degraded or down.
    Implements exponential backoff for 429 responses.
    """
    def __init__(self, providers: list[tuple[str, ProviderHealth]]):
        self.providers = providers   # ordered by preference

    def call(self, prompt: str, model: str) -> tuple[str, str]:
        """Returns (output, provider_used)."""
        for provider_name, health in self.providers:
            if health.status == "down":
                print(f"  [Router] ⛔  {provider_name} is DOWN — skipping")
                continue
            
            print(f"  [Router] → Trying {provider_name}/{model}...", end=" ")
            time.sleep(0.03)
            
            # Simulate provider response
            if random.random() < health.error_rate:
                if health.status == "degraded":
                    print(f"429 Rate Limited — failing over")
                    continue
                print(f"503 Service Error — failing over")
                continue
            
            print(f"✅  Success ({health.p99_latency_ms:.0f}ms)")
            return f"Response from {provider_name}", provider_name
        
        raise RuntimeError("All providers exhausted — circuit breaker open")

random.seed(42)
print("🔄  Provider Failover Demo")
print("=" * 60)

health_map = {
    "openai":    ProviderHealth("openai",    "down",     0.80, 200),
    "anthropic": ProviderHealth("anthropic", "degraded", 0.40, 350),
    "google":    ProviderHealth("google",    "healthy",  0.02, 180),
}

router2 = ReliableRouter([
    ("openai",    health_map["openai"]),
    ("anthropic", health_map["anthropic"]),
    ("google",    health_map["google"]),
])

output, used = router2.call("Summarise this incident report", "gpt-4o")
print(f"\n  Final response from: {used}")
print(f"  Output: {output}")


🔄  Provider Failover Demo
  [Router] ⛔  openai is DOWN — skipping
  [Router] → Trying anthropic/gpt-4o... 429 Rate Limited — failing over
  [Router] → Trying google/gpt-4o... ✅  Success (180ms)

  Final response from: google
  Output: Response from google
